In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import io
from PIL import Image
from folium.plugins import HeatMap
import folium
import os
from PIL import Image
from io import BytesIO
import requests
import time


# PATHS SETUP

figures_path = os.path.join('figures')

monthly_metrics_path = 'data/monthly_metrics.csv'
observations_path = 'data/observations.csv'
new_species_path = 'data/new_species_photo.csv'
taxon_counts_path = 'data/taxon_counts.csv'

# DATA LOADING

df_monthly = pd.read_csv(monthly_metrics_path)
df_new_species = pd.read_csv(new_species_path)
df_taxon_count = pd.read_csv(taxon_counts_path)
df_observations = pd.read_csv(observations_path)

# COLOR PALETTES

palette_orange = ['#f85532'] 
palette_blue = ['#2b2e4f']




# 1.Graficación y muestra de métricas principales

1.1. número de observaciones, número de observadores, identificadores y especies
1.2. variación respecto al mes anterior

Estas métricas se mostrarán en forma de tabla debido a que es la mejor opción disponible 

# 2.Graficación evolución mensual de las metricas principales

Gráficos por observaciones, observadores, idntificadores y especies

In [ ]:
def plot_monthly_metrics(df_monthly, column: str):
    # Configuración de la figura
    plt.figure(figsize=(12, 7))
    
    # Crear el gráfico de barras
    bars = plt.bar(df_monthly['month'], 
                  df_monthly[column], 
                  color='#f85532')
    
    # Añadir los valores encima de las barras
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., 
                 height * 1.02,
                 f'{height}',
                 ha='center', 
                 va='bottom')
    
    

    # Personalización del gráfico
    plt.title(f'Evolució mensual de {column}', pad=20, fontsize=14)
    plt.xlabel('Mes', labelpad=10)
    plt.ylabel(column, labelpad=10)
    
    plt.xticks(rotation=45, ha='right', rotation_mode='anchor') 

    # Ajustar el rango del eje Y
    plt.ylim(0, max(df_monthly[column]) * 1.15)

    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['left'].set_visible(False)
    plt.gca().yaxis.set_ticks_position('none')
    
    # Ajustar márgenes
    plt.tight_layout()

    monthly_dir = f'{figures_path}/monthly_metrics'
    os.makedirs(monthly_dir, exist_ok=True)
    img_path = os.path.join(monthly_dir, f'monthly_{column}.png')
    plt.savefig(img_path, dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
plot_monthly_metrics(df_monthly, 'species')
plot_monthly_metrics(df_monthly, 'observations')
plot_monthly_metrics(df_monthly, 'observers')
plot_monthly_metrics(df_monthly, 'identifiers')

# 3. Graficación taxonomias

Graficación nuevas especies 

In [ ]:
def plot_new_species(df_new_species):

        image_col = 'photos_medium_url'     
        name_col = 'taxon_name'          
        date_col = 'observed_on'           
        user_col = 'user_login'
        attribution_col = 'attribution'
        url_col = 'obs_url'
        url_photo_col = 'photos_medium_url'

        plot_dir = 'figures/photos_new_species'
        os.makedirs(plot_dir, exist_ok=True)
    
    # Contador para numerar las imágenes
        counter = 1
        # Mostrar las imágenes
        for index, row in df_new_species.iterrows():
            
                time.sleep(1)
                
                response = requests.get(row[image_col], timeout=10)
                img = Image.open(BytesIO(response.content))

                fig, ax = plt.subplots(figsize=(5, 5))
                ax.imshow(img)
                ax.axis('off')
                #plt.figure(figsize=(5, 5))
                #plt.imshow(img)
                #plt.axis('off')
                fig.suptitle(f"Nom de l'espècie: {row[name_col]} \n Usuari: {row[user_col]} \n Observat el {row[date_col]}", fontsize = 12, ha='center')
                fig.text(0.5,0.01, f'{row[attribution_col]}', ha='right', fontsize=8, style = 'italic')               
                plt.tight_layout()
                plt.show()

                #img_path = os.path.join(plot_dir, f"new_species_{counter:03d}.png")
                #fig.savefig(img_path, bbox_inches='tight', dpi=300)
                #plt.close()

                attribution_filename = f"new_species_{counter:03d}.txt"
                attribution_path = os.path.join(plot_dir, attribution_filename)
                with open(attribution_path, 'w', encoding='utf-8') as f:
                    f.write(row[attribution_col])

                # Guardar URL (.txt)
                url_filename = f"new_species_{counter:03d}_url.txt"
                url_path = os.path.join(plot_dir, url_filename)
                with open(url_path, 'w', encoding='utf-8') as f:
                    f.write(row[url_col])

                #url_photo = f"new_species_{counter:03d}_photo_url.txt"
                #url_path_photo = os.path.join(plot_dir, url_photo)
                #with open(url_path_photo, 'w', encoding='utf-8') as f:
                    #f.write(row[url_photo_col])

                counter += 1

In [ ]:
plot_new_species( df_new_species )

Top 10 taxon ranks 

In [ ]:
def plot_top_species(df_taxon_count, taxon_rank: str):
    top_df = df_taxon_count[df_taxon_count['taxon_rank'] == taxon_rank]\
             .sort_values('count', ascending=False)\
             .head(10)
    
    plt.figure(figsize=(10, 6))
    bars = plt.barh(top_df['taxon_name'], top_df['count'], color='#f85532')
    
    # Añadir etiquetas con formato
    for bar in bars:
        width = bar.get_width()
        plt.text(width + (0.01 * top_df['count'].max()), 
                bar.get_y() + bar.get_height()/2,
                f'{width:,}',
                va='center')
    
    plt.title(f'Top {taxon_rank} més observats')
    plt.xlabel('Número d\'observacions')
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['left'].set_visible(False)
    plt.gca().yaxis.set_ticks_position('none')  
    plt.gca().invert_yaxis()
    plt.tight_layout()
    
    # Guardar la figura
    plot_dir = 'figures/taxon_plots'
    os.makedirs(plot_dir, exist_ok=True)
    img_path = os.path.join(plot_dir, f'top_{taxon_rank}.png')
    plt.savefig(img_path, dpi=300, bbox_inches='tight')
    plt.close()
    
  

In [ ]:
plot_top_species(df_taxon_count, 'kingdom')
plot_top_species(df_taxon_count, 'phylum')
plot_top_species(df_taxon_count, 'class')
plot_top_species(df_taxon_count, 'species')
plot_top_species(df_taxon_count, 'family')

# 5. Mapa calor para densidad de observaciones

Crear la función que toma un dataframe con el formato de df_obs (con esos nombres de columna, "latitude", "longitude") y lo mapee en el mapa de calor. Ese dataframe puede estar con las observaciones totales, filtrato por un kingdom, por un usuario, por un mes, o por lo que sea, pero no le afecta a la función, que lo hará siempre igual sobre un dataframe con las mismas columnas.

In [ ]:
def get_heatmap(iconic_taxon = None):

    df_valid = df_observations.dropna(subset=["latitude", "longitude"])
    
    if iconic_taxon is not None:
        df_valid = df_valid[df_valid["iconic_taxon"] == iconic_taxon]
        if df_valid.empty:
            print(f"No hay datos disponibles para el taxón: {iconic_taxon}")
            return
    
    heat_data = df_valid[["latitude", "longitude"]].values.tolist()

    mean_lat = df_valid["latitude"].mean()
    mean_lon = df_valid["longitude"].mean()
   

    m = folium.Map(location=[mean_lat, mean_lon], zoom_start=11)
    HeatMap(heat_data).add_to(m)

    os.makedirs("figures", exist_ok=True)

    html_path = f"figures/heatmap_plots/heatmap_{iconic_taxon}.html"
    m.save(html_path)

    img_data = m._to_png(3)
    img = Image.open(io.BytesIO(img_data))
    
    img.save(f'figures/heatmap_plots/heatmap_image_{iconic_taxon}.png')

In [ ]:
get_heatmap()
get_heatmap('plantae')
get_heatmap('fungi')
get_heatmap('animalia')
get_heatmap('aves')